# 01 — Télécharger les prix (Yahoo Finance / FRED)

Ce notebook télécharge les prix actions/indices/FX (Yahoo) et taux (FRED, optionnel) et crée un **`prices.csv`** au format `timestamp,asset,price`.

In [ ]:
# Installer les dépendances (exécuter une seule fois par environnement)
!pip install -q yfinance pandas numpy fredapi

In [ ]:
import os, pandas as pd, numpy as np
import yfinance as yf
from datetime import datetime

# --- Paramètres à adapter ---
START = "2020-01-01"
END   = datetime.utcnow().strftime("%Y-%m-%d")  # aujourd'hui
INTERVAL = "1d"  # ou "1h" pour intraday

# Mapping label -> ticker Yahoo
STOCKS  = {"AAPL":"AAPL", "MSFT":"MSFT"}
INDICES = {"SPX":"^GSPC", "CAC40":"^FCHI"}
FX      = {"EURUSD":"EURUSD=X", "USDJPY":"USDJPY=X"}

OUT_CSV = "prices.csv"  # sortie unifiée


In [ ]:
def fetch_yf(map_label2tick, start, end, interval):
    if not map_label2tick:
        return pd.DataFrame(columns=["timestamp","asset","price"])
    labels_by_ticker = {v:k for k,v in map_label2tick.items()}
    data = yf.download(list(map_label2tick.values()), start=start, end=end, interval=interval, auto_adjust=False, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"]
    else:
        close = data["Close"] if "Close" in data.columns else data
    if isinstance(close, pd.Series):
        close = close.to_frame()
    close = close.rename(columns=lambda c: labels_by_ticker.get(c, c))
    out = close.stack().reset_index()
    out.columns = ["timestamp","asset","price"]
    return out.dropna(subset=["price"])

In [ ]:
# Téléchargement Yahoo
frames = []
for mapping in (STOCKS, INDICES, FX):
    df = fetch_yf(mapping, START, END, INTERVAL)
    frames.append(df)
prices = pd.concat(frames, ignore_index=True)
prices = prices.sort_values(["asset","timestamp"]).reset_index(drop=True)
prices.to_csv(OUT_CSV, index=False)
print(f"Sauvé {len(prices):,} lignes dans {OUT_CSV}")
prices.head()

### (Optionnel) Ajouter des taux FRED
Créez une clé API gratuite sur FRED, puis :

```bash
export FRED_API_KEY=VOTRE_CLE
```


In [ ]:
# Optionnel : ajouter des séries FRED
USE_FRED = False  # passez à True si vous voulez ajouter des taux
FRED_SERIES = ["DGS10","DGS2"]

if USE_FRED:
    try:
        from fredapi import Fred
        fred = Fred(api_key=os.environ.get("FRED_API_KEY",""))
        fred_frames = []
        for sid in FRED_SERIES:
            s = fred.get_series(sid, observation_start=START, observation_end=END)
            df = s.to_frame(name="price")
            df["timestamp"] = df.index.tz_localize(None)
            df["asset"] = sid
            fred_frames.append(df[["timestamp","asset","price"]])
        if fred_frames:
            fred_df = pd.concat(fred_frames, ignore_index=True)
            prices = pd.concat([prices, fred_df], ignore_index=True)
            prices = prices.sort_values(["asset","timestamp"]).reset_index(drop=True)
            prices.to_csv(OUT_CSV, index=False)
            print(f"Ajouté FRED: total {len(prices):,} lignes → {OUT_CSV}")
    except Exception as e:
        print("FRED indisponible:", e)